## 1. External label coverage (image IDs only)
Which split does each expert-label / NEATX file cover? Only the image-ID column is read (`usecols`); no label columns are loaded, so this check reveals nothing about test labels.

In [1]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from dotenv import load_dotenv

from src.paths import SPLIT_CSV

load_dotenv()
DATA_ROOT = Path(os.getenv("DATA_ROOT"))

split = pd.read_csv(SPLIT_CSV, usecols=["Image Index", "split"]).set_index("Image Index")["split"]

EXP = DATA_ROOT / "NIH_expert_labels"
files = {
    "expert all-findings labels": (EXP / "nih-cxr-14_labels_all_findings_expert_labels_test_labels.csv", "Image ID"),
    "expert all-findings readers": (EXP / "nih-cxr-14_labels_all_findings_expert_labels_test_individual_readers.csv", "Image ID"),
    "expert four-findings 'test'": (EXP / "nih-cxr-14_labels_four_findings_expert_labels_test_labels.csv", "Image Index"),
    "expert four-findings 'validation'": (EXP / "nih-cxr-14_labels_four_findings_expert_labels_validation_labels.csv", "Image Index"),
    "expert four-findings readers": (EXP / "nih-cxr-14_labels_four_findings_expert_labels_individual_readers.csv", "Image ID"),
    "NEATX drains": (DATA_ROOT / "NEATX" / "NIH-CX14_TubeAnnotations_NonExperts_aggregated.csv", "Image Index"),
}

rows = []
for name, (path, id_col) in files.items():
    ids = pd.read_csv(path, usecols=[id_col])[id_col].drop_duplicates()
    counts = ids.map(split).value_counts(dropna=False)
    rows.append({"file": name, "images": len(ids), **counts.to_dict()})

pd.DataFrame(rows).set_index("file").fillna(0).astype(int)[["images", "train", "val", "test"]]

,images,train,val,test
file,,,,
expert all-findings labels,810,0,0,810
expert all-findings readers,810,0,0,810
expert four-findings 'test',1962,0,0,1962
expert four-findings 'validation',2414,2148,266,0
expert four-findings readers,4376,2148,266,1962
NEATX drains,3543,1564,219,1760


## Coverage -> access rules
- Expert all-findings labels + readers (810 imgs): **100% test** -> closed until the single test run.
- Expert four-findings "test" (1,962): **100% test** -> closed until the test run.
- Expert four-findings "validation" (2,414): 2,148 in our **train**, 266 in our **val** -> usable for dev (label-noise estimate on train; dry run of expert re-scoring on val). Google's "validation" is NOT our val.
- Four-findings individual readers (4,376): mixes all splits -> load only through a dev-filtering loader.
- NEATX drains (3,543): 1,564 train / 219 val / **1,760 test** -> load only through a dev-filtering loader; build the drain analysis on val, run once on test.
- Val-side samples are small (266 expert, 219 NEATX): for building/debugging analyses, not for claims.